# TCN Hyperparameter Optimization — GWO (Grey Wolf Optimizer)

**Author:** Ibrahim Hanafy  
**Date:** August 2026  
**Objective:** Hyperparameter optimization using Grey Wolf Optimizer (mealpy) for TCN ECG forecasting.  

**Reduced Search Space (3 params):**
| Parameter | Type | Range |
|-----------|------|-------|
| `n_filters` | categorical | {32, 64, 128, 256} |
| `dropout` | continuous | 0.05–0.40 |
| `lr` | continuous (log) | 1e-4 – 1e-2 |

**Fixed:** n_blocks=4, kernel_size=3, batch_size=128  
**Horizon:** H=10 only  

---

## 1 — Imports & Setup

In [1]:
!pip install wfdb -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.3 MB/s eta 0:00:00


In [2]:
import os, time, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [3]:
# ── TensorFlow Setup ─────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, Add, Activation, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

TensorFlow 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [4]:
# ── MealPy Setup ──────────────────────────────────────────────────────────────
!pip install -q mealpy
from mealpy import GWO, FloatVar, IntegerVar

import mealpy
print(f'MealPy {mealpy.__version__}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 20.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
wfdb 4.3.1 requires numpy>=1.26.4, but you have numpy 1.26.0 which is incompatible.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
tpot 1.1.0 requires numpy>=1.26.4, but you have numpy 1.26.0 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.

## 2 — Constants & Dataset

In [5]:
# ── Dataset path ──────────────────────────────────────────────────────────────
DATA_DIR = r"/kaggle/input/datasets/rracer17/mit-bih-mitdb/mit-bih-arrhythmia-database-1.0.0"

# ── Paper constants (Section 3) ──────────────────────────────────────────────
FS           = 360
TOTAL_STEPS  = 100_000
TRAIN_STEPS  = 40_000
VAL_STEPS    = 10_000
TEST_STEPS   = 50_000
LOOKBACK     = 10

# ── All 21 patients (excluding 111 & 118) ───────────────────────────────────
PATIENTS = [
    '100', '101', '102', '103', '104', '105',
    '106', '107', '108', '109', '112', '113',
    '114', '115', '116', '117', '119',
    '121', '122', '123', '124'
]
assert len(PATIENTS) == 21

# ── Single horizon ───────────────────────────────────────────────────────────
HORIZONS = [10]

# ── Pilot patients for HPO ───────────────────────────────────────────────────
PILOT_PATIENTS = ['101', '103', '108', '119', '124']
PILOT_HORIZON  = 10

# ── Fixed hyperparameters ────────────────────────────────────────────────────
FIXED_N_BLOCKS    = 3
FIXED_KERNEL_SIZE = 3
FIXED_BATCH_SIZE  = 128

print(f'Full patients : {len(PATIENTS)}')
print(f'Pilot patients: {len(PILOT_PATIENTS)} → {PILOT_PATIENTS}')
print(f'Horizon       : H={PILOT_HORIZON}')
print(f'Fixed         : n_blocks={FIXED_N_BLOCKS}, kernel={FIXED_KERNEL_SIZE}, batch={FIXED_BATCH_SIZE}')

Full patients : 21
Pilot patients: 5 → ['101', '103', '108', '119', '124']
Horizon       : H=10
Fixed         : n_blocks=3, kernel=3, batch=128


## 3 — Data Pipeline

Identical to baseline notebooks — load, split, normalise, create sequences.

In [6]:
def load_ecg_signal(record_id, data_dir, n_steps=100_000):
    """Load MLII lead from MIT-BIH, truncate/pad to n_steps."""
    path = os.path.join(data_dir, record_id)
    rec  = wfdb.rdrecord(path)
    sig_names_upper = [s.upper() for s in rec.sig_name]
    ch = sig_names_upper.index('MLII') if 'MLII' in sig_names_upper else 0
    signal = rec.p_signal[:, ch].astype(np.float32)
    if len(signal) < n_steps:
        pad = np.full(n_steps - len(signal), signal[-1], dtype=np.float32)
        signal = np.concatenate([signal, pad])
    return signal[:n_steps]


def preprocess_patient(signal, train_steps=40_000, val_steps=10_000):
    """Split → train/val/test, MinMax-normalise (fit on train only)."""
    train_end = train_steps
    val_end   = train_steps + val_steps
    train_raw = signal[:train_end]
    val_raw   = signal[train_end:val_end]
    test_raw  = signal[val_end:]
    scaler     = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train_raw.reshape(-1, 1)).flatten()
    val_norm   = scaler.transform(val_raw.reshape(-1, 1)).flatten()
    test_norm  = scaler.transform(test_raw.reshape(-1, 1)).flatten()
    return train_norm, val_norm, test_norm, scaler


def make_multistep_sequences(signal, lookback, horizon):
    """X = lookback window, y = next H steps (Multi-Output)."""
    X, y = [], []
    for i in range(len(signal) - lookback - horizon + 1):
        X.append(signal[i : i + lookback])
        y.append(signal[i + lookback : i + lookback + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def compute_metrics(y_true, y_pred):
    yt, yp = y_true.flatten(), y_pred.flatten()
    return {
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'MAE':  float(mean_absolute_error(yt, yp)),
        'R2':   float(r2_score(yt, yp)),
    }


def compute_per_step_metrics(y_true, y_pred):
    rows = []
    for h in range(y_true.shape[1]):
        m = compute_metrics(y_true[:, h], y_pred[:, h])
        m['Step'] = h + 1
        rows.append(m)
    return pd.DataFrame(rows)


print('Data pipeline functions defined.')

Data pipeline functions defined.


In [7]:
# ── Load ALL patients ─────────────────────────────────────────────────────────
patient_signals = {}

print(f'Loading {len(PATIENTS)} patients...\n')
for rid in tqdm(PATIENTS, desc='Patients'):
    signal = load_ecg_signal(rid, DATA_DIR, n_steps=TOTAL_STEPS)
    tr, vl, te, sc = preprocess_patient(signal, TRAIN_STEPS, VAL_STEPS)
    patient_signals[rid] = {'train': tr, 'val': vl, 'test': te, 'scaler': sc}
    tqdm.write(f'  Patient {rid:>3s} | train={len(tr):,}  val={len(vl):,}  test={len(te):,}')

print(f'\nAll {len(patient_signals)} patients loaded.')

Loading 21 patients...



Patients:   0%|          | 0/21 [00:00<?, ?it/s]

  Patient 100 | train=40,000  val=10,000  test=50,000
  Patient 101 | train=40,000  val=10,000  test=50,000
  Patient 102 | train=40,000  val=10,000  test=50,000
  Patient 103 | train=40,000  val=10,000  test=50,000
  Patient 104 | train=40,000  val=10,000  test=50,000
  Patient 105 | train=40,000  val=10,000  test=50,000
  Patient 106 | train=40,000  val=10,000  test=50,000
  Patient 107 | train=40,000  val=10,000  test=50,000
  Patient 108 | train=40,000  val=10,000  test=50,000
  Patient 109 | train=40,000  val=10,000  test=50,000
  Patient 112 | train=40,000  val=10,000  test=50,000
  Patient 113 | train=40,000  val=10,000  test=50,000
  Patient 114 | train=40,000  val=10,000  test=50,000
  Patient 115 | train=40,000  val=10,000  test=50,000
  Patient 116 | train=40,000  val=10,000  test=50,000
  Patient 117 | train=40,000  val=10,000  test=50,000
  Patient 119 | train=40,000  val=10,000  test=50,000
  Patient 121 | train=40,000  val=10,000  test=50,000
  Patient 122 | train=40,000

## 4 — Model Architecture

Same residual block as baseline. `build_tcn` accepts all hyperparameters as arguments.

In [8]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(out)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1)(x)
    return Add()([x, out])


def build_tcn(lookback, output_size, n_blocks, n_filters, kernel_size, dropout_rate, learning_rate):
    """Build TCN with explicit hyperparameters (no globals)."""
    inp = Input(shape=(lookback, 1))
    x = inp
    for i in range(n_blocks):
        x = residual_block(x, n_filters, kernel_size, 2 ** i, dropout_rate)
    x = x[:, -1, :]  # last time-step
    x = Dense(n_filters, activation='relu')(x)
    out = Dense(output_size)(x)
    model = Model(inp, out, name=f'TCN_out{output_size}')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss='mse')
    return model


print('Model architecture defined.')

Model architecture defined.


## 5 — GWO Objective Function

**Strategy:**
- Train on 5 pilot patients at H=10
- 100 epochs max with patience=20
- Objective = mean RMSE across pilot patients

**Search space (3 params):**
- `n_filters` index: integer 0–3 → maps to {32, 64, 128, 256}
- `dropout`: continuous 0.05–0.40
- `lr`: continuous in log10 space (−4 to −2) → 1e-4 to 1e-2

In [9]:
FILTER_OPTIONS = [32, 64, 128, 256]

# Track all evaluations for analysis
gwo_eval_log = []


def objective(solution):
    """GWO objective: decode solution → train TCN on pilot patients → return mean RMSE."""
    # ── Decode solution vector ────────────────────────────────────────────────
    filter_idx = int(np.clip(np.round(solution[0]), 0, 3))
    n_filters  = FILTER_OPTIONS[filter_idx]
    dropout    = float(np.clip(solution[1], 0.05, 0.40))
    lr         = float(10 ** np.clip(solution[2], -4, -2))  # log10 scale

    eval_id = len(gwo_eval_log) + 1
    print(f'  [Eval {eval_id:3d}] filters={n_filters}, dropout={dropout:.3f}, lr={lr:.6f}', end=' ')

    rmse_scores = []

    for pid in PILOT_PATIENTS:
        tf.keras.backend.clear_session()
        gc.collect()

        tr = patient_signals[pid]['train']
        vl = patient_signals[pid]['val']
        te = patient_signals[pid]['test']

        X_tr, y_tr = make_multistep_sequences(tr, LOOKBACK, PILOT_HORIZON)
        X_vl, y_vl = make_multistep_sequences(vl, LOOKBACK, PILOT_HORIZON)
        X_te, y_te = make_multistep_sequences(te, LOOKBACK, PILOT_HORIZON)

        X_tr_r = X_tr.reshape(-1, X_tr.shape[1], 1)
        X_vl_r = X_vl.reshape(-1, X_vl.shape[1], 1)
        X_te_r = X_te.reshape(-1, X_te.shape[1], 1)

        model = build_tcn(LOOKBACK, PILOT_HORIZON,
                          FIXED_N_BLOCKS, n_filters, FIXED_KERNEL_SIZE,
                          dropout, lr)

        history = model.fit(
            X_tr_r, y_tr,
            validation_data=(X_vl_r, y_vl),
            epochs=100,
            batch_size=FIXED_BATCH_SIZE,
            callbacks=[
                EarlyStopping(monitor='val_loss', patience=20,
                              restore_best_weights=True),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                  patience=10, min_lr=1e-6),
            ],
            verbose=0
        )

        preds = model.predict(X_te_r, batch_size=2048, verbose=0)
        rmse = float(np.sqrt(mean_squared_error(y_te.flatten(), preds.flatten())))
        rmse_scores.append(rmse)

        del model
        gc.collect()

    mean_rmse = float(np.mean(rmse_scores))
    print(f'→ RMSE={mean_rmse:.6f}')

    gwo_eval_log.append({
        'eval': eval_id,
        'n_filters': n_filters,
        'dropout': round(dropout, 4),
        'lr': round(lr, 8),
        'mean_rmse': mean_rmse,
    })

    return mean_rmse


print('Objective function defined.')
print(f'Each eval trains on {len(PILOT_PATIENTS)} patients × H={PILOT_HORIZON}')
print(f'Fixed: n_blocks={FIXED_N_BLOCKS}, kernel={FIXED_KERNEL_SIZE}, batch={FIXED_BATCH_SIZE}')

Objective function defined.
Each eval trains on 5 patients × H=10
Fixed: n_blocks=3, kernel=3, batch=128


## 6 — Run GWO Search

**Budget:** pop_size × epoch total evaluations.  
**No pruning** — GWO evaluates every wolf fully each iteration.  
**No crash recovery** — if kernel dies, re-run from scratch.

In [10]:
# ┌──────────────────────────────────────────────────────────────────────────┐
# │  GWO CONFIGURATION                                                     │
# └──────────────────────────────────────────────────────────────────────────┘
POP_SIZE  = 8      # wolves per iteration
N_EPOCHS  = 4      # GWO iterations (total evals ≈ pop_size × epoch)

print(f'GWO config: {POP_SIZE} wolves × {N_EPOCHS} epochs')
print(f'Expected total evaluations: ~{POP_SIZE * N_EPOCHS}')
print(f'Estimated time: ~{POP_SIZE * N_EPOCHS * 3:.0f}–{POP_SIZE * N_EPOCHS * 8:.0f} min')

GWO config: 8 wolves × 4 epochs
Expected total evaluations: ~32
Estimated time: ~96–256 min


In [11]:
# ── Define search space ──────────────────────────────────────────────────────
bounds = [
    IntegerVar(lb=0, ub=3, name="n_filters_idx"),    # index → {32, 64, 128, 256}
    FloatVar(lb=0.05, ub=0.40, name="dropout"),       # continuous
    FloatVar(lb=-4.0, ub=-2.0, name="log10_lr"),      # log scale: 10^x
]

problem = {
    "bounds": bounds,
    "obj_func": objective,
    "minmax": "min",
    "log_to": "console",
}

# ── Run GWO ──────────────────────────────────────────────────────────────────
print(f'\n{"=" * 70}')
print(f'Starting GWO optimization...')
print(f'{"=" * 70}\n')

t0 = time.time()
optimizer = GWO.OriginalGWO(epoch=N_EPOCHS, pop_size=POP_SIZE)
best_agent = optimizer.solve(problem)
elapsed = time.time() - t0

# ── Decode best solution ─────────────────────────────────────────────────────
best_filter_idx = int(np.clip(np.round(best_agent.solution[0]), 0, 3))
best_n_filters  = FILTER_OPTIONS[best_filter_idx]
best_dropout    = float(np.clip(best_agent.solution[1], 0.05, 0.40))
best_lr         = float(10 ** np.clip(best_agent.solution[2], -4, -2))
best_rmse       = best_agent.target.fitness

print(f'\n{"=" * 70}')
print(f'GWO Search Complete!')
print(f'  Total time    : {elapsed/60:.1f} min')
print(f'  Total evals   : {len(gwo_eval_log)}')
print(f'  Best RMSE     : {best_rmse:.6f}')
print(f'  Best params   :')
print(f'    n_filters   = {best_n_filters}')
print(f'    dropout     = {best_dropout:.4f}')
print(f'    lr          = {best_lr:.8f}')
print(f'{"=" * 70}')

2026/08/17 04:06:54 AM, INFO, mealpy.swarm_based.GWO.OriginalGWO: OriginalGWO(epoch=4, pop_size=8)



Starting GWO optimization...

  [Eval   1] filters=64, dropout=0.122, lr=0.000518 

I0000 00:00:1786939615.834170      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786939615.837239      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786939628.924599      89 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


→ RMSE=0.037007
  [Eval   2] filters=256, dropout=0.375, lr=0.000808 → RMSE=0.036035
  [Eval   3] filters=128, dropout=0.199, lr=0.006396 → RMSE=0.036016
  [Eval   4] filters=64, dropout=0.389, lr=0.001297 → RMSE=0.038907
  [Eval   5] filters=256, dropout=0.162, lr=0.008330 → RMSE=0.034700
  [Eval   6] filters=64, dropout=0.211, lr=0.007880 → RMSE=0.034594
  [Eval   7] filters=32, dropout=0.146, lr=0.002246 → RMSE=0.037925
  [Eval   8] filters=128, dropout=0.071, lr=0.007352 → RMSE=0.034359
  [Eval   9] filters=128, dropout=0.234, lr=0.010000 → RMSE=0.034945
  [Eval  10] filters=128, dropout=0.260, lr=0.005512 → RMSE=0.036123
  [Eval  11] filters=128, dropout=0.158, lr=0.010000 → RMSE=0.035786
  [Eval  12] filters=64, dropout=0.226, lr=0.002376 → RMSE=0.036166
  [Eval  13] filters=64, dropout=0.064, lr=0.004721 → RMSE=0.032657
  [Eval  14] filters=128, dropout=0.103, lr=0.010000 → RMSE=0.033758
  [Eval  15] filters=256, dropout=0.170, lr=0.010000 → RMSE=0.034621
  [Eval  16] filters=12

2026/08/17 07:40:13 AM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 1, Current best: 0.03265703174280256, Global best: 0.03265703174280256, Runtime: 6448.68314 seconds


→ RMSE=0.035441
  [Eval  17] filters=128, dropout=0.164, lr=0.010000 → RMSE=0.035901
  [Eval  18] filters=128, dropout=0.101, lr=0.005154 → RMSE=0.035019
  [Eval  19] filters=128, dropout=0.050, lr=0.001531 → RMSE=0.031206
  [Eval  20] filters=128, dropout=0.063, lr=0.010000 → RMSE=0.030703
  [Eval  21] filters=128, dropout=0.081, lr=0.006416 → RMSE=0.034486
  [Eval  22] filters=128, dropout=0.091, lr=0.010000 → RMSE=0.034392
  [Eval  23] filters=128, dropout=0.050, lr=0.001331 → RMSE=0.032543
  [Eval  24] filters=128, dropout=0.066, lr=0.002675 

2026/08/17 09:24:28 AM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 2, Current best: 0.030703310395242205, Global best: 0.030703310395242205, Runtime: 6255.35149 seconds


→ RMSE=0.031133
  [Eval  25] filters=128, dropout=0.067, lr=0.002513 → RMSE=0.031467
  [Eval  26] filters=128, dropout=0.061, lr=0.010000 → RMSE=0.031298
  [Eval  27] filters=64, dropout=0.059, lr=0.002349 → RMSE=0.031125
  [Eval  28] filters=128, dropout=0.063, lr=0.000930 → RMSE=0.031536
  [Eval  29] filters=128, dropout=0.056, lr=0.002259 → RMSE=0.030622
  [Eval  30] filters=128, dropout=0.058, lr=0.001674 → RMSE=0.031755
  [Eval  31] filters=64, dropout=0.060, lr=0.003372 → RMSE=0.035983
  [Eval  32] filters=128, dropout=0.055, lr=0.006860 

2026/08/17 11:09:31 AM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 3, Current best: 0.030621850820684388, Global best: 0.030621850820684388, Runtime: 6302.17574 seconds


→ RMSE=0.034235
  [Eval  33] filters=128, dropout=0.059, lr=0.003758 

## 7 — Analyse Search Results

In [ ]:
# ── Evaluation log DataFrame ─────────────────────────────────────────────────
df_evals = pd.DataFrame(gwo_eval_log)
df_evals = df_evals.sort_values('mean_rmse')
print('All evaluations (sorted by RMSE):\n')
display(df_evals)

In [ ]:
# ── Convergence plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: All evaluations over time
ax = axes[0]
ax.scatter(df_evals.sort_values('eval')['eval'],
           df_evals.sort_values('eval')['mean_rmse'],
           alpha=0.6, s=40, c='steelblue')
ax.set_xlabel('Evaluation #')
ax.set_ylabel('Mean RMSE')
ax.set_title('All GWO Evaluations')
ax.grid(alpha=0.3)

# Plot 2: Best-so-far convergence
ax = axes[1]
df_sorted = df_evals.sort_values('eval')
best_so_far = df_sorted['mean_rmse'].cummin()
ax.plot(df_sorted['eval'], best_so_far, 'o-', color='#1565C0', linewidth=2)
ax.set_xlabel('Evaluation #')
ax.set_ylabel('Best RMSE So Far')
ax.set_title('GWO Convergence')
ax.grid(alpha=0.3)

plt.suptitle('Grey Wolf Optimizer — Search Progress', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Parameter distributions (scatter colored by RMSE) ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, col, label in zip(axes,
    ['n_filters', 'dropout', 'lr'],
    ['n_filters', 'dropout', 'learning rate']):
    sc = ax.scatter(df_evals[col], df_evals['mean_rmse'],
                    c=df_evals['mean_rmse'], cmap='RdYlGn_r', s=60, alpha=0.8)
    ax.set_xlabel(label)
    ax.set_ylabel('Mean RMSE')
    ax.set_title(f'RMSE vs {label}')
    ax.grid(alpha=0.3)
    if col == 'lr':
        ax.set_xscale('log')

plt.suptitle('Parameter vs RMSE', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8 — Full Evaluation with Best Config

Run best GWO hyperparameters on **all 21 patients** at **H=10**.  
200 epochs, patience 50 (full training budget).

In [ ]:
# ┌──────────────────────────────────────────────────────────────────────────┐
# │  BEST CONFIG FROM GWO SEARCH                                           │
# └──────────────────────────────────────────────────────────────────────────┘
CONFIG_NAME   = 'GWO_Optimized'
EPOCHS        = 200
PATIENCE      = 50
BATCH_SIZE    = FIXED_BATCH_SIZE
NUM_FILTERS   = best_n_filters
KERNEL_SIZE   = FIXED_KERNEL_SIZE
NUM_BLOCKS    = FIXED_N_BLOCKS
DROPOUT_RATE  = best_dropout
LEARNING_RATE = best_lr

CSV_NAME = 'tcn_mo_gwo_optimized_results.csv'
PLOT_DIR = 'plots_gwo_optimized'
os.makedirs(PLOT_DIR, exist_ok=True)

print(f'Config    : {CONFIG_NAME}')
print(f'Patients  : {len(PATIENTS)}')
print(f'Horizons  : {HORIZONS}')
print(f'TCN       : blocks={NUM_BLOCKS}, filters={NUM_FILTERS}, kernel={KERNEL_SIZE}')
print(f'Dropout   : {DROPOUT_RATE:.4f}')
print(f'Training  : epochs={EPOCHS}, patience={PATIENCE}, batch={BATCH_SIZE}, lr={LEARNING_RATE:.6f}')

In [ ]:
# ── Full Experiment Loop ─────────────────────────────────────────────────────
results          = []
per_step_results = {}
saved_preds      = {}
training_histories = {}

total_runs = len(PATIENTS) * len(HORIZONS)
print(f'Running: {len(PATIENTS)} patients × {len(HORIZONS)} horizons = {total_runs} models')
print('=' * 80)

for rid in tqdm(PATIENTS, desc='Patients'):
    tr = patient_signals[rid]['train']
    vl = patient_signals[rid]['val']
    te = patient_signals[rid]['test']

    for horizon in HORIZONS:
        X_tr, y_tr = make_multistep_sequences(tr, LOOKBACK, horizon)
        X_vl, y_vl = make_multistep_sequences(vl, LOOKBACK, horizon)
        X_te, y_te = make_multistep_sequences(te, LOOKBACK, horizon)

        t0 = time.time()
        model = build_tcn(LOOKBACK, horizon,
                          NUM_BLOCKS, NUM_FILTERS, KERNEL_SIZE,
                          DROPOUT_RATE, LEARNING_RATE)

        X_tr_r = X_tr.reshape(-1, X_tr.shape[1], 1)
        X_vl_r = X_vl.reshape(-1, X_vl.shape[1], 1)

        history = model.fit(
            X_tr_r, y_tr,
            validation_data=(X_vl_r, y_vl),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[
                EarlyStopping(monitor='val_loss', patience=PATIENCE,
                              restore_best_weights=True),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                  patience=max(PATIENCE // 2, 2), min_lr=1e-6),
            ],
            verbose=0
        )

        preds = model.predict(
            X_te.reshape(-1, X_te.shape[1], 1),
            batch_size=2048, verbose=0
        )
        elapsed = time.time() - t0

        m = compute_metrics(y_te, preds)
        m['Time_s'] = round(elapsed, 2)
        results.append({'Patient': rid, 'Horizon': horizon, **m})

        if horizon > 1:
            per_step_results[(rid, horizon)] = compute_per_step_metrics(y_te, preds)

        saved_preds[(rid, horizon)] = (y_te.copy(), preds.copy())
        training_histories[(rid, horizon)] = {
            'loss': history.history['loss'],
            'val_loss': history.history['val_loss'],
        }

        del model
        gc.collect()

        tqdm.write(
            f'  Patient {rid} H={horizon:2d} | '
            f'R²={m["R2"]:.4f}  RMSE={m["RMSE"]:.4f}  MAE={m["MAE"]:.4f}  '
            f'({elapsed:.1f}s)'
        )

    tf.keras.backend.clear_session()
    gc.collect()

print(f'\nAll {len(results)} experiments complete.')

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────────
df_results = pd.DataFrame(results)
df_results.to_csv(CSV_NAME, index=False)
print(f'Results saved to {CSV_NAME}')
display(df_results)

## 9 — Compare: Baseline vs GWO Optimized

In [ ]:
# ── Load baseline results for comparison ─────────────────────────────────────
df_baseline = pd.read_csv('Figures/results/tcn_mo_ours_config_results.csv')
df_optimized = df_results.copy()

# Filter baseline to H=10 only for comparison
print('Comparison: Baseline (Ours) vs GWO Optimized — H=10')
print('=' * 70)

for h in HORIZONS:
    b = df_baseline[df_baseline.Horizon == h]
    o = df_optimized[df_optimized.Horizon == h]
    if len(b) == 0:
        print(f'  H={h}: no baseline data')
        continue
    rmse_b, rmse_o = b.RMSE.mean(), o.RMSE.mean()
    r2_b, r2_o     = b.R2.mean(), o.R2.mean()
    mae_b, mae_o   = b.MAE.mean(), o.MAE.mean()
    rmse_pct = (rmse_o - rmse_b) / rmse_b * 100
    r2_pct   = (r2_o - r2_b) / r2_b * 100

    print(f'  RMSE : {rmse_b:.6f} → {rmse_o:.6f} ({rmse_pct:+.1f}%)')
    print(f'  MAE  : {mae_b:.6f} → {mae_o:.6f}')
    print(f'  R²   : {r2_b:.4f} → {r2_o:.4f} ({r2_pct:+.2f}%)')

print('=' * 70)
print('Negative RMSE Δ% = improvement (lower is better)')

In [ ]:
# ── Per-patient comparison ───────────────────────────────────────────────────
h = 10
b = df_baseline[df_baseline.Horizon == h].sort_values('Patient')
o = df_optimized[df_optimized.Horizon == h].sort_values('Patient')

if len(b) > 0:
    wins = (o.RMSE.values < b.RMSE.values).sum()
    total = len(b)
    print(f'H={h}: GWO wins {wins}/{total} patients ({wins/total*100:.0f}%)')

    # Bar chart
    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(b))
    w = 0.35
    ax.bar(x - w/2, b.RMSE.values, w, label='Baseline', color='#90CAF9', edgecolor='white')
    ax.bar(x + w/2, o.RMSE.values, w, label='GWO Optimized', color='#1565C0', edgecolor='white')
    ax.set_xlabel('Patient')
    ax.set_ylabel('RMSE')
    ax.set_title(f'Per-Patient RMSE — Baseline vs GWO Optimized (H={h})')
    ax.set_xticks(x)
    ax.set_xticklabels(b.Patient.values, rotation=45)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/comparison_baseline_vs_gwo_H{h}.png', dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Wilcoxon signed-rank test ────────────────────────────────────────────────
from scipy.stats import wilcoxon

h = 10
b_rmse = df_baseline[df_baseline.Horizon == h].sort_values('Patient').RMSE.values
o_rmse = df_optimized[df_optimized.Horizon == h].sort_values('Patient').RMSE.values

if len(b_rmse) == len(o_rmse) and len(b_rmse) > 0:
    stat, p = wilcoxon(b_rmse, o_rmse)
    sig = '✓ significant' if p < 0.05 else '✗ not significant'
    print(f'Wilcoxon test (H={h}): p={p:.4f} → {sig}')

## 10 — Summary

In [ ]:
print('\n' + '=' * 80)
print('GWO HYPERPARAMETER OPTIMIZATION COMPLETE')
print('=' * 80)
print(f'\nOptimizer   : Grey Wolf Optimizer (mealpy)')
print(f'Search space: 3 params (n_filters, dropout, lr)')
print(f'Fixed       : n_blocks={FIXED_N_BLOCKS}, kernel={FIXED_KERNEL_SIZE}, batch={FIXED_BATCH_SIZE}')
print(f'Population  : {POP_SIZE} wolves × {N_EPOCHS} epochs = {len(gwo_eval_log)} evals')
print(f'\nBest Configuration:')
print(f'  n_filters   = {best_n_filters}')
print(f'  dropout     = {best_dropout:.4f}')
print(f'  lr          = {best_lr:.8f}')
print(f'  Pilot RMSE  = {best_rmse:.6f}')
print(f'\nFull Evaluation ({CONFIG_NAME}):')
print(f'  Patients    : {len(PATIENTS)}')
print(f'  Horizon     : {HORIZONS}')
print(f'  Results CSV : {CSV_NAME}')
print(f'  Plots       : {PLOT_DIR}/')
print(f'\nMean RMSE (all patients, H=10): {df_results.RMSE.mean():.6f}')
print(f'Mean R²   (all patients, H=10): {df_results.R2.mean():.4f}')
print('=' * 80)